# 03 — Agent IA 1 V3
## Choix autonome et itératif de la fenêtre d'analyse

### Objectif scientifique

Dans les versions précédentes, l'agent choisissait une ou plusieurs fenêtres dans une
liste préparée à l'avance, par exemple :

```text
[25, 50, 100]
```

Cette V3 change le problème.

L'agent reçoit :

- la caractéristique recherchée ;
- une description sémantique de cette caractéristique ;
- un résumé de la trace GPX ;
- les outils de détection déterministes disponibles.

Il choisit ensuite **lui-même** :

```text
window_m
step_m
```

dans un espace continu.

Il ne choisit donc plus parmi trois propositions.

### Pourquoi un agent IA ici ?

Le choix d'une bonne échelle dépend du phénomène recherché.

```text
forte pente
≠
montée progressive
≠
succession de lacets
```

Une règle fixe unique est donc peu satisfaisante.

L'agent peut :

1. interpréter sémantiquement ce que l'on cherche ;
2. proposer une échelle d'observation ;
3. appeler le détecteur Python ;
4. observer le résumé des résultats ;
5. accepter ou réviser son choix.

C'est cette **boucle décision → outil → observation → révision** qui justifie
le terme d'agent IA.

Le LLM ne calcule toujours aucune géométrie.

In [ ]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import requests

## 1. Charger la route normalisée du notebook 02 V2

In [ ]:
ROUTE_PATH = Path("data/output/route_resampled.csv")

if not ROUTE_PATH.exists():
    raise FileNotFoundError(
        "Exécute d'abord le notebook 02 V2."
    )

route = pd.read_csv(ROUTE_PATH)

required = {"distance_m", "lat", "lon", "elev_m"}
missing = required - set(route.columns)

if missing:
    raise ValueError(
        f"Colonnes manquantes : {sorted(missing)}"
    )

BASE_SPACING_M = float(
    np.median(
        np.diff(route["distance_m"].to_numpy(dtype=float))[:-1]
    )
    if len(route) > 2
    else 10.0
)

ROUTE_LENGTH_M = float(route["distance_m"].iloc[-1])

print("Route :", ROUTE_PATH)
print("Longueur :", round(ROUTE_LENGTH_M, 1), "m")
print("Résolution de base :", round(BASE_SPACING_M, 3), "m")

## 2. Caractéristiques et outils disponibles

Les bornes ci-dessous ne sont **pas des choix proposés au LLM**.

Ce sont uniquement des garde-fous pour empêcher des valeurs absurdes comme :

```text
fenêtre = 0.1 m
fenêtre = 20 km sur une trace de 5 km
```

À l'intérieur de ces bornes, l'agent peut choisir librement n'importe quelle valeur.

In [ ]:
FEATURES = {
    "forte_pente": {
        "family": "slope_intensity",
        "detector": "slope",
        "description": (
            "Identifier des portions où la pente est localement forte. "
            "Le phénomène peut être court et doit conserver les variations marquées."
        ),
        "window_bounds_m": (10.0, 250.0),
    },

    "pente_douce": {
        "family": "slope_intensity",
        "detector": "slope",
        "description": (
            "Identifier des portions de pente modérée et relativement progressive, "
            "sans confondre une oscillation locale avec une tendance réelle."
        ),
        "window_bounds_m": (10.0, 400.0),
    },

    "montee": {
        "family": "slope_direction",
        "detector": "slope",
        "description": (
            "Identifier une tendance de montée significative le long de l'itinéraire. "
            "On cherche davantage une tendance spatiale qu'une micro-variation."
        ),
        "window_bounds_m": (20.0, 600.0),
    },

    "descente": {
        "family": "slope_direction",
        "detector": "slope",
        "description": (
            "Identifier une tendance de descente significative le long de l'itinéraire. "
            "On cherche davantage une tendance spatiale qu'une micro-variation."
        ),
        "window_bounds_m": (20.0, 600.0),
    },

    "lacets": {
        "family": "geometry",
        "detector": "switchbacks",
        "description": (
            "Identifier une succession rapprochée de changements de direction alternés "
            "caractéristiques de lacets. La fenêtre doit être assez grande pour observer "
            "plusieurs virages, mais pas au point de mélanger des zones indépendantes."
        ),
        "window_bounds_m": (30.0, 800.0),
    },
}

## 3. Résumé de la trace fourni à l'agent

L'agent ne reçoit pas tous les points bruts.

On lui fournit un résumé compact utile pour son choix d'échelle.

In [ ]:
def build_route_summary(route):
    d = route["distance_m"].to_numpy(dtype=float)
    z = route["elev_m"].to_numpy(dtype=float)

    dz = np.diff(z)
    dd = np.diff(d)

    valid = dd > 0

    local_slopes = np.zeros_like(dz, dtype=float)
    local_slopes[valid] = 100.0 * dz[valid] / dd[valid]

    return {
        "route_length_m": float(d[-1]),
        "base_spacing_m": float(
            np.median(dd[valid])
            if valid.any()
            else 10.0
        ),
        "elevation_min_m": float(np.min(z)),
        "elevation_max_m": float(np.max(z)),
        "elevation_range_m": float(np.max(z) - np.min(z)),
        "median_abs_base_slope_pct": float(
            np.median(np.abs(local_slopes[valid]))
            if valid.any()
            else 0.0
        ),
        "p90_abs_base_slope_pct": float(
            np.percentile(np.abs(local_slopes[valid]), 90)
            if valid.any()
            else 0.0
        ),
        "n_points": int(len(route)),
    }


ROUTE_SUMMARY = build_route_summary(route)

print(
    json.dumps(
        ROUTE_SUMMARY,
        ensure_ascii=False,
        indent=2,
    )
)

## 4. Détecteur déterministe de pente

L'agent choisit l'échelle.

Python calcule la pente.

In [ ]:
def compute_window_slopes(
    route,
    window_m,
    step_m,
):
    distances = route["distance_m"].to_numpy(dtype=float)
    elevations = route["elev_m"].to_numpy(dtype=float)

    rows = []
    start = 0.0
    total = float(distances[-1])

    while start + window_m <= total + 1e-9:
        end = min(start + window_m, total)

        if end - start < window_m - 1e-6:
            break

        z0 = np.interp(
            start,
            distances,
            elevations,
        )

        z1 = np.interp(
            end,
            distances,
            elevations,
        )

        slope_pct = (
            100.0
            * (z1 - z0)
            / (end - start)
        )

        rows.append({
            "start_m": float(start),
            "end_m": float(end),
            "slope_pct": float(slope_pct),
        })

        start += step_m

    return pd.DataFrame(rows)


def detect_slope_feature(
    route,
    feature,
    window_m,
    step_m,
):
    slopes = compute_window_slopes(
        route,
        window_m=window_m,
        step_m=step_m,
    )

    if slopes.empty:
        return slopes

    abs_slope = slopes["slope_pct"].abs()

    if feature == "forte_pente":
        mask = abs_slope >= 12.0

    elif feature == "pente_douce":
        mask = (
            (abs_slope >= 3.0)
            & (abs_slope < 8.0)
        )

    elif feature == "montee":
        mask = slopes["slope_pct"] >= 5.0

    elif feature == "descente":
        mask = slopes["slope_pct"] <= -5.0

    else:
        raise ValueError(feature)

    out = slopes.loc[mask].copy()

    if out.empty:
        return out

    out["feature"] = feature
    out["family"] = FEATURES[feature]["family"]
    out["resolution_m"] = float(window_m)

    out["confidence"] = np.clip(
        abs(out["slope_pct"]) / 20.0,
        0.0,
        1.0,
    )

    return out

## 5. Détecteur déterministe de lacets

In [ ]:
def bearing_deg(
    lat1,
    lon1,
    lat2,
    lon2,
):
    phi1 = math.radians(float(lat1))
    phi2 = math.radians(float(lat2))

    dlambda = math.radians(
        float(lon2) - float(lon1)
    )

    y = (
        math.sin(dlambda)
        * math.cos(phi2)
    )

    x = (
        math.cos(phi1)
        * math.sin(phi2)
        - math.sin(phi1)
        * math.cos(phi2)
        * math.cos(dlambda)
    )

    return (
        math.degrees(
            math.atan2(y, x)
        )
        + 360.0
    ) % 360.0


def angular_difference(a, b):
    return (
        b - a + 180.0
    ) % 360.0 - 180.0


def detect_switchbacks(
    route,
    window_m,
    step_m,
    min_turn_deg=35.0,
    min_turns=3,
):
    d = route["distance_m"].to_numpy(dtype=float)
    lat = route["lat"].to_numpy(dtype=float)
    lon = route["lon"].to_numpy(dtype=float)

    if len(route) < 4:
        return pd.DataFrame()

    bearings = np.array([
        bearing_deg(
            lat[i],
            lon[i],
            lat[i + 1],
            lon[i + 1],
        )
        for i in range(len(route) - 1)
    ])

    turns = np.array([
        angular_difference(
            bearings[i],
            bearings[i + 1],
        )
        for i in range(len(bearings) - 1)
    ])

    turn_distance = d[1:-1]

    rows = []
    start = 0.0
    total = float(d[-1])

    while start + window_m <= total + 1e-9:
        end = start + window_m

        idx = (
            (turn_distance >= start)
            & (turn_distance <= end)
        )

        local_turns = turns[idx]

        significant = local_turns[
            np.abs(local_turns)
            >= min_turn_deg
        ]

        signs = np.sign(significant)

        alternations = (
            int(
                np.sum(
                    signs[1:] * signs[:-1] < 0
                )
            )
            if len(signs) >= 2
            else 0
        )

        if (
            len(significant) >= min_turns
            and alternations >= 2
        ):
            confidence = min(
                1.0,
                0.2 * len(significant)
                + 0.15 * alternations,
            )

            rows.append({
                "feature": "lacets",
                "family": "geometry",
                "start_m": float(start),
                "end_m": float(end),
                "resolution_m": float(window_m),
                "confidence": float(confidence),
                "direction_changes": int(
                    len(significant)
                ),
                "alternations": int(
                    alternations
                ),
            })

        start += step_m

    return pd.DataFrame(rows)

## 6. Exécuter un outil de détection

Cette fonction constitue l'interface entre l'agent et les outils Python.

In [ ]:
def run_detector(
    feature,
    window_m,
    step_m,
):
    detector = FEATURES[feature]["detector"]

    if detector == "slope":
        return detect_slope_feature(
            route,
            feature,
            window_m,
            step_m,
        )

    if detector == "switchbacks":
        return detect_switchbacks(
            route,
            window_m,
            step_m,
        )

    raise ValueError(
        f"Détecteur inconnu : {detector}"
    )

## 7. Résumer les observations d'un outil

L'agent ne reçoit pas un énorme tableau de milliers de lignes après chaque essai.

Il reçoit seulement un résumé lui permettant d'évaluer son choix.

In [ ]:
def summarize_detection(
    feature,
    result,
    window_m,
    step_m,
):
    if result.empty:
        return {
            "window_m": float(window_m),
            "step_m": float(step_m),
            "n_detections": 0,
            "detected_span_sum_m": 0.0,
            "approx_route_coverage_ratio": 0.0,
            "mean_confidence": 0.0,
        }

    intervals = result[
        ["start_m", "end_m"]
    ].sort_values("start_m")

    merged = []

    for _, row in intervals.iterrows():
        start = float(row["start_m"])
        end = float(row["end_m"])

        if not merged or start > merged[-1][1]:
            merged.append([start, end])
        else:
            merged[-1][1] = max(
                merged[-1][1],
                end,
            )

    detected_length = sum(
        end - start
        for start, end in merged
    )

    summary = {
        "window_m": float(window_m),
        "step_m": float(step_m),
        "n_detections": int(len(result)),
        "n_merged_zones": int(len(merged)),
        "detected_span_sum_m": float(
            detected_length
        ),
        "approx_route_coverage_ratio": float(
            detected_length
            / max(ROUTE_LENGTH_M, 1e-9)
        ),
        "mean_confidence": float(
            result["confidence"].mean()
        ),
    }

    if "slope_pct" in result.columns:
        summary.update({
            "mean_abs_detected_slope_pct": float(
                result["slope_pct"].abs().mean()
            ),
            "max_abs_detected_slope_pct": float(
                result["slope_pct"].abs().max()
            ),
        })

    if "direction_changes" in result.columns:
        summary.update({
            "mean_direction_changes": float(
                result["direction_changes"].mean()
            ),
            "max_direction_changes": int(
                result["direction_changes"].max()
            ),
        })

    return summary

## 8. Client Ollama

Le modèle répond en JSON structuré.

La justification demandée est une **justification courte du choix d'échelle**,
pas une chaîne de raisonnement détaillée.

In [ ]:
OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL = "llama3.1:8b"


def ask_ollama_json(prompt):
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL,
            "messages": [
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            "stream": False,
            "format": "json",
            "options": {
                "temperature": 0,
            },
        },
        timeout=120,
    )

    if response.status_code != 200:
        raise RuntimeError(
            f"Ollama HTTP "
            f"{response.status_code}: "
            f"{response.text}"
        )

    content = response.json()[
        "message"
    ]["content"]

    return json.loads(content)

## 9. Valider un choix libre de l'agent

L'agent choisit librement sa valeur.

Python impose seulement :

```text
min <= window <= max
BASE_SPACING_M <= step <= window
```

afin de garantir une exécution physiquement et numériquement raisonnable.

In [ ]:
def validate_scale_choice(
    feature,
    window_m,
    step_m,
):
    minimum, maximum = (
        FEATURES[feature][
            "window_bounds_m"
        ]
    )

    # Une fenêtre ne peut pas dépasser
    # la longueur réelle du trajet.
    effective_max = min(
        float(maximum),
        float(ROUTE_LENGTH_M),
    )

    window = float(window_m)
    step = float(step_m)

    window = max(
        float(minimum),
        min(window, effective_max),
    )

    step = max(
        float(BASE_SPACING_M),
        min(step, window),
    )

    return window, step

## 10. Première décision autonome de l'agent

In [ ]:
def propose_initial_scale(
    feature,
):
    cfg = FEATURES[feature]

    minimum, maximum = (
        cfg["window_bounds_m"]
    )

    effective_max = min(
        maximum,
        ROUTE_LENGTH_M,
    )

    prompt = f'''
Tu es l'agent IA chargé de choisir l'échelle spatiale d'analyse d'un itinéraire.

Caractéristique recherchée :
{feature}

Description sémantique :
{cfg["description"]}

Détecteur Python disponible :
{cfg["detector"]}

Résumé de l'itinéraire :
{json.dumps(ROUTE_SUMMARY, ensure_ascii=False, indent=2)}

Tu dois choisir toi-même une fenêtre d'analyse et un pas de déplacement.

Il n'existe PAS de liste de fenêtres candidates.

Contraintes de sécurité uniquement :
- window_m doit être compris entre {minimum:.1f} et {effective_max:.1f} mètres ;
- step_m doit être compris entre {BASE_SPACING_M:.1f} et window_m ;
- choisis des valeurs cohérentes avec la nature spatiale du phénomène ;
- le pas peut être inférieur à la fenêtre afin d'obtenir des observations chevauchantes ;
- tu ne calcules aucune pente ni aucun virage : Python le fera.

Réponds uniquement en JSON sous cette forme :
{{
  "window_m": 83.0,
  "step_m": 17.0,
  "justification": "courte justification du choix d'échelle"
}}
'''

    decision = ask_ollama_json(
        prompt
    )

    window, step = validate_scale_choice(
        feature,
        decision["window_m"],
        decision["step_m"],
    )

    return {
        "window_m": float(window),
        "step_m": float(step),
        "justification": str(
            decision.get(
                "justification",
                "",
            )
        ),
    }

## 11. Révision après observation

Après exécution du détecteur, l'agent reçoit :

- son choix précédent ;
- le résumé des détections obtenues ;
- l'historique de ses essais.

Il peut répondre :

```text
accept
```

ou :

```text
revise
```

et proposer une nouvelle échelle.

C'est cette boucle qui transforme réellement le choix de fenêtre en comportement agentique.

In [ ]:
def review_scale(
    feature,
    current_decision,
    detection_summary,
    history,
):
    cfg = FEATURES[feature]

    minimum, maximum = (
        cfg["window_bounds_m"]
    )

    effective_max = min(
        maximum,
        ROUTE_LENGTH_M,
    )

    prompt = f'''
Tu es un agent IA qui règle l'échelle spatiale d'un détecteur géographique.

Caractéristique :
{feature}

Description :
{cfg["description"]}

Résumé de la route :
{json.dumps(ROUTE_SUMMARY, ensure_ascii=False, indent=2)}

Choix actuel :
{json.dumps(current_decision, ensure_ascii=False, indent=2)}

Résumé du résultat obtenu :
{json.dumps(detection_summary, ensure_ascii=False, indent=2)}

Historique des essais :
{json.dumps(history, ensure_ascii=False, indent=2)}

Décide si cette échelle est raisonnable pour observer le phénomène.

Tu n'as pas à maximiser artificiellement le nombre de détections.
Une absence de détection peut être légitime.

Si le choix paraît spatialement cohérent, accepte-le.
Sinon, révise librement window_m et step_m.

Contraintes de sécurité :
- window_m dans [{minimum:.1f}, {effective_max:.1f}] ;
- step_m dans [{BASE_SPACING_M:.1f}, window_m].

Réponds uniquement en JSON :

si tu acceptes :
{{
  "action": "accept",
  "justification": "courte justification"
}}

si tu révises :
{{
  "action": "revise",
  "window_m": 91.0,
  "step_m": 19.0,
  "justification": "courte justification"
}}
'''

    return ask_ollama_json(
        prompt
    )

## 12. Boucle complète de l'Agent IA

Le nombre d'itérations est limité pour éviter une boucle infinie.

Le choix final n'est **jamais** extrait d'une liste prédéfinie.

In [ ]:
MAX_AGENT_ITERATIONS = 3


def run_scale_agent(
    feature,
    max_iterations=MAX_AGENT_ITERATIONS,
):
    decision = propose_initial_scale(
        feature
    )

    history = []

    for iteration in range(
        1,
        max_iterations + 1,
    ):
        window = decision["window_m"]
        step = decision["step_m"]

        print(
            f"\n{feature} — itération {iteration}"
        )
        print(
            "Fenêtre :",
            round(window, 2),
            "m | pas :",
            round(step, 2),
            "m",
        )
        print(
            "Justification :",
            decision.get(
                "justification",
                "",
            ),
        )

        result = run_detector(
            feature,
            window,
            step,
        )

        summary = summarize_detection(
            feature,
            result,
            window,
            step,
        )

        history.append({
            "iteration": int(iteration),
            "window_m": float(window),
            "step_m": float(step),
            "detection_summary": summary,
        })

        print(
            "Résumé détecteur :",
            json.dumps(
                summary,
                ensure_ascii=False,
            ),
        )

        if iteration == max_iterations:
            print(
                "Nombre maximal d'itérations atteint."
            )

            return {
                "feature": feature,
                "final_decision": decision,
                "history": history,
                "result": result,
                "accepted_by_agent": False,
            }

        review = review_scale(
            feature,
            decision,
            summary,
            history,
        )

        action = str(
            review.get(
                "action",
                "accept",
            )
        ).lower()

        print(
            "Décision de révision :",
            action,
        )

        print(
            "Justification :",
            review.get(
                "justification",
                "",
            ),
        )

        if action == "accept":
            return {
                "feature": feature,
                "final_decision": decision,
                "history": history,
                "result": result,
                "accepted_by_agent": True,
            }

        if action != "revise":
            raise ValueError(
                f"Action Agent inconnue : {action}"
            )

        new_window, new_step = (
            validate_scale_choice(
                feature,
                review["window_m"],
                review["step_m"],
            )
        )

        decision = {
            "window_m": float(
                new_window
            ),
            "step_m": float(
                new_step
            ),
            "justification": str(
                review.get(
                    "justification",
                    "",
                )
            ),
        }

    raise RuntimeError(
        "Boucle Agent terminée de façon inattendue."
    )

## 13. Lancer l'Agent IA sur les caractéristiques

Pour commencer, on conserve les cinq caractéristiques actuelles.

À terme, cette liste pourra elle aussi devenir un objet de décision agentique.

In [ ]:
FEATURES_TO_ANALYZE = [
    "forte_pente",
    "pente_douce",
    "montee",
    "descente",
    "lacets",
]

agent_runs = {}

for feature in FEATURES_TO_ANALYZE:
    print("\n" + "=" * 80)
    print("AGENT —", feature)

    agent_runs[feature] = (
        run_scale_agent(feature)
    )

## 14. Inspecter les fenêtres choisies par l'Agent

Cette table est importante pour les expériences futures.

Elle permettra notamment de comparer :

```text
fenêtre choisie par l'agent
VS
fenêtre considérée pertinente par un expert
VS
qualité finale de détection
```

In [ ]:
scale_rows = []

for feature, run in agent_runs.items():
    final = run["final_decision"]

    scale_rows.append({
        "feature": feature,
        "detector": FEATURES[
            feature
        ]["detector"],
        "window_m": float(
            final["window_m"]
        ),
        "step_m": float(
            final["step_m"]
        ),
        "n_iterations": int(
            len(run["history"])
        ),
        "accepted_by_agent": bool(
            run[
                "accepted_by_agent"
            ]
        ),
        "justification": final.get(
            "justification",
            "",
        ),
    })

agent_scale_df = pd.DataFrame(
    scale_rows
)

display(agent_scale_df)

## 15. Standardiser les annotations atomiques

La sortie reste compatible avec l'algorithme du notebook 04.

Chaque observation conserve notamment :

```text
resolution_m = fenêtre réellement choisie par l'Agent IA
```

In [ ]:
STANDARD_COLUMNS = {
    "feature",
    "family",
    "start_m",
    "end_m",
    "resolution_m",
    "confidence",
}

atomic_annotations = []

for feature, run in agent_runs.items():
    result = run["result"]

    if result.empty:
        continue

    final_decision = (
        run["final_decision"]
    )

    for _, row in result.iterrows():
        metrics = {
            "agent_window_m": float(
                final_decision[
                    "window_m"
                ]
            ),
            "agent_step_m": float(
                final_decision[
                    "step_m"
                ]
            ),
            "agent_iterations": int(
                len(run["history"])
            ),
            "agent_accepted_scale": bool(
                run[
                    "accepted_by_agent"
                ]
            ),
        }

        for column in result.columns:
            if column in STANDARD_COLUMNS:
                continue

            value = row[column]

            if pd.isna(value):
                continue

            if hasattr(value, "item"):
                value = value.item()

            metrics[column] = value

        atomic_annotations.append({
            "feature": str(
                row["feature"]
            ),
            "family": str(
                row["family"]
            ),
            "start_m": float(
                row["start_m"]
            ),
            "end_m": float(
                row["end_m"]
            ),
            "resolution_m": float(
                row["resolution_m"]
            ),
            "confidence": float(
                row["confidence"]
            ),
            "metrics": metrics,
        })

print(
    "Annotations atomiques :",
    len(atomic_annotations)
)

## 16. Export

Trois sorties sont produites :

```text
agent1_annotations_atomic.json
agent1_annotations_atomic.csv
agent1_scale_decisions.json
```

Le troisième fichier conserve l'historique du raisonnement opérationnel de l'agent
au niveau **décision / outil / observation / révision**.

In [ ]:
OUTPUT_DIR = Path(
    "data/output"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

json_path = (
    OUTPUT_DIR
    / "agent1_annotations_atomic.json"
)

csv_path = (
    OUTPUT_DIR
    / "agent1_annotations_atomic.csv"
)

decision_path = (
    OUTPUT_DIR
    / "agent1_scale_decisions.json"
)

with json_path.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        atomic_annotations,
        f,
        ensure_ascii=False,
        indent=2,
    )

pd.DataFrame([
    {
        **{
            k: v
            for k, v in annotation.items()
            if k != "metrics"
        },
        "metrics_json": json.dumps(
            annotation["metrics"],
            ensure_ascii=False,
        ),
    }
    for annotation in atomic_annotations
]).to_csv(
    csv_path,
    index=False,
)

serializable_runs = {}

for feature, run in agent_runs.items():
    serializable_runs[feature] = {
        "final_decision": (
            run["final_decision"]
        ),
        "history": run["history"],
        "accepted_by_agent": bool(
            run[
                "accepted_by_agent"
            ]
        ),
    }

with decision_path.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        serializable_runs,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("Exports :")
print("-", json_path)
print("-", csv_path)
print("-", decision_path)

# Ce que cette V3 change scientifiquement

### Ancienne approche

```text
Humain :
"voici [25, 50, 100]"

LLM :
"je choisis 50"
```

Le LLM faisait essentiellement de la sélection dans un petit menu.

### Nouvelle approche

```text
caractéristique + contexte spatial
             ↓
          Agent IA
             ↓
choisit librement fenêtre + pas
             ↓
       détecteur Python
             ↓
     observation du résultat
             ↓
          Agent IA
        accepte / révise
```

Le rôle de l'agent devient donc défendable :

> **adapter dynamiquement la granularité d'analyse au phénomène spatial recherché,
> en utilisant les résultats des outils comme rétroaction.**

### À expérimenter ensuite

- mesurer la stabilité des fenêtres choisies ;
- répéter le même itinéraire plusieurs fois ;
- comparer plusieurs modèles ;
- comparer le choix agent à une grille exhaustive de fenêtres ;
- déterminer si le choix agent améliore réellement précision / rappel ;
- à terme, permettre une **fenêtre différente selon la portion de l'itinéraire**.